# 03 - Frozen-model test evaluation

Evaluate the lexical baseline and, optionally, frozen instruction/adapter models on the same fictional test split. Do not use the test errors to tune the next training run and still call the test held out.

The data and labels are fictional teaching fixtures. Scores do not reproduce the original paper or validate a psychological assessment.

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

from empathy_lab.baseline import fit_baseline, predict_baseline
from empathy_lab.config import ModelConfig
from empathy_lab.data import load_dataset
from empathy_lab.evaluation import evaluate

parents = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in parents if (p / 'data' / 'fictional_support.jsonl').is_file()), None)
if ROOT is None:
    ROOT = next((p / 'workshops' / 'leadership-evidence-lab' for p in parents
                 if (p / 'workshops' / 'leadership-evidence-lab' / 'data' / 'fictional_support.jsonl').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the multi-lora repository.')
PROFILE = 'qwen-local'
RUN_MODEL_EVALUATION = False
RUN_NAME = 'manual-01'
ADAPTER = ROOT / 'artifacts' / PROFILE / RUN_NAME / 'adapter'
records = load_dataset(ROOT / 'data' / 'fictional_support.jsonl')
training = [row for row in records if row['split'] == 'train']
test = [row for row in records if row['split'] == 'test']
print('Test records:', len(test))

In [ ]:
baseline = fit_baseline(training)
baseline_predictions = predict_baseline(baseline, test)
reports = {'lexical_baseline': asdict(evaluate(test, baseline_predictions))}
print(json.dumps(reports, indent=2))

## Interpret metrics and denominators

Read per-mechanism macro-F1 with fixed classes 0, 1, 2, balanced accuracy, and confusion matrices. Always pair valid-only metrics with invalid-response counts and coverage. A model that emits one correct response and nine malformed responses must not appear equivalent to a model that answers every example.

Inspect disagreements against the fictional author rubric. These small, patterned fixtures can reward lexical shortcuts; they cannot establish population-level performance, fairness, or empathy as an internal mental state.

In [ ]:
if RUN_MODEL_EVALUATION:
    from empathy_lab.modeling import predict
    config = ModelConfig.from_json(ROOT / 'configs' / f'{PROFILE}.json')
    base_predictions = predict(config, test)
    reports['instruction_model'] = asdict(evaluate(test, base_predictions))
    if not (ADAPTER / 'adapter_config.json').is_file():
        raise FileNotFoundError('Train the configured adapter in notebook 02 before comparing it.')
    adapted_predictions = predict(config, test, adapter_path=ADAPTER)
    reports['lora_model'] = asdict(evaluate(test, adapted_predictions))
    print(json.dumps(reports, indent=2))
else:
    print('GPU/model test runs disabled. Enable only after freezing validation choices.')

In [ ]:
for row in test:
    prediction = baseline_predictions[row['id']]
    print(json.dumps({'id': row['id'], 'expected': row['labels'], 'predicted': prediction}, ensure_ascii=False))

## Report and next phases

Record model/revision, seed, fixture fingerprint, prompt, training budget, hardware, timing, valid/invalid counts, and all per-mechanism results. Distinguish observed runs from prepared Qwen/Nemotron profiles.

Do not interpret the delta from lexical baseline to LoRA as improvement over EPITOME's original RoBERTa model. No original data, annotation validation, or matched protocol was used.

Next: broaden independently annotated fictional scenarios, test rationale/uncertainty behavior, then consider separately authorized research. Leadership mapping and Hermes optimization come later. Clear outputs before publication.